In [ ]:
import pandas as pd
import json

# Load the data
df = pd.read_csv('data-wvs.csv')
with open('map-wvs.json', 'r') as f:
    column_map = json.load(f)

# Initial inspection
print(df.head())
print(df['B_COUNTRY_ALPHA'].unique())
print(f"Dataframe shape: {df.shape}")

In [ ]:
# Identify Argentina's row
arg_data = df[df['B_COUNTRY_ALPHA'] == 'Argentina'].iloc[0]

# Numeric columns only
numeric_cols = df.select_dtypes(include=['number']).columns
df_numeric = df[numeric_cols]

# Calculate means and stds for all countries
means = df_numeric.mean()
stds = df_numeric.std()

# Calculate z-scores for Argentina
arg_zscores = (arg_data[numeric_cols] - means) / stds

# Get top 5 positive and top 5 negative deviations (excluding NaN)
arg_zscores = arg_zscores.dropna()
top_high = arg_zscores.sort_values(ascending=False).head(10)
top_low = arg_zscores.sort_values(ascending=True).head(10)

# Translate top deviations
def get_label(q_code):
    return column_map.get(q_code, q_code)

print("Top deviations (High relative to global mean):")
for code, z in top_high.items():
    print(f"{code}: {get_label(code)} (z={z:.2f})")

print("\nTop deviations (Low relative to global mean):")
for code, z in top_low.items():
    print(f"{code}: {get_label(code)} (z={z:.2f})")

# Also look at specific key metrics
key_metrics = ['Q46', 'Q1', 'Q250', 'Q252', 'Q57', 'Q188', 'Q190', 'Q240']
print("\nKey Metrics for Argentina:")
for k in key_metrics:
    val = arg_data.get(k, None)
    avg = means.get(k, None)
    if val is not None:
        print(f"{k} ({get_label(k)}): Arg={val:.2f}, Global Avg={avg:.2f}")

In [ ]:
# Check mean and Argentina's value for a few key columns to determine direction
cols_to_check = ['Q56', 'Q250', 'Q145', 'Q132', 'Q52', 'Q24', 'Q20', 'Q190']
check_df = pd.DataFrame({
    'Argentina': arg_data[cols_to_check],
    'Global_Mean': means[cols_to_check],
    'Z-Score': arg_zscores[cols_to_check],
    'Label': [get_label(c) for c in cols_to_check]
})
print(check_df)

In [ ]:
# Read the map for Q56 and a few others
target_qs = ['Q56', 'Q89', 'Q70', 'Q250', 'Q145', 'Q132', 'Q24']
for q in target_qs:
    print(f"{q}: {column_map.get(q)}")

In [ ]:
# Check min/max values to infer scales
print(df[target_qs].describe().loc[['min', 'max']])